In [ ]:
# %pip install xgboost


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import display
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier
%matplotlib inline

In [ ]:
def preprocess_beacon_rssi(df_raw, window=3):
    """
    4つのRSSI列に対して個人差を抑える前処理を行う。
    - 元値
    - 個人内 min-max 正規化（行方向）
    - 列方向 z-score 正規化（個人差除去）
    - 行方向比率
    - 行方向順位
    - 短期移動平均
    - 短期差分
    """
    rssi_cols = ["810257F7","81025919","81025B89","810B3B76"]
    X = df_raw[rssi_cols].copy()

    # --- ① 個人内 min-max 正規化（行方向）---
    X_norm = (X - X.min()) / (X.max() - X.min()).replace(0, np.nan)
    X_norm = X_norm.fillna(0)
    X_norm.columns = [c+"_norm" for c in rssi_cols]

    # --- ② 列方向 z-score 正規化（個人差除去）---
    X_colnorm = (X - X.mean()) / X.std().replace(0, np.nan)
    X_colnorm = X_colnorm.fillna(0)
    X_colnorm.columns = [c+"_colnorm" for c in rssi_cols]

    # --- ③ 行方向比率 ---
    X_ratio = X.div(X.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    X_ratio.columns = [c+"_ratio" for c in rssi_cols]

    # --- ④ 行方向順位 ---
    X_rank = X.rank(axis=1, method='dense', pct=True)
    X_rank.columns = [c+"_rank" for c in rssi_cols]

    # --- ⑤ 移動平均（短期平滑化）---
    X_ma = X.rolling(window, min_periods=1).mean()
    X_ma.columns = [c+"_ma" for c in rssi_cols]

    # --- ⑥ 差分（短期変化）---
    X_diff = X.diff().fillna(0)
    X_diff.columns = [c+"_diff" for c in rssi_cols]

    # 結合して返す
    df_proc = pd.concat([X, X_norm, X_colnorm, X_ratio, X_rank, X_ma, X_diff], axis=1)
    # df_proc = pd.concat([X,X_colnorm, X_ratio, X_rank], axis=1)

    return df_proc


In [ ]:
# データの読み込み
df = pd.read_csv(f"../../../../実験/label_beacon/tuji2_beacon_label_0~6.csv")
df_label = df[['beacon_label']].copy()
df = preprocess_beacon_rssi(df)

In [ ]:
df.describe()

# ランダムフォレスト用関数

In [ ]:
def load_and_rondam_forest_by_keyword(keyword, way = "all",heatmap=True):

    # データの読み込み
    df = pd.read_csv(f"../../../../実験/label_beacon/{keyword}_beacon_label_0~6.csv")
    df_label = df[['beacon_label']].copy()

    df = preprocess_beacon_rssi(df)

    if way == "all":
        X = df
    elif way == "rssi":
        X = df[["810257F7","81025919", "81025B89","810B3B76"]]
    
    
    # 特徴量とラベルの分割
    y = df_label['beacon_label']

    # 訓練データとテストデータの分割
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # ランダムフォレストモデルの作成と訓練
    model = RandomForestClassifier(n_estimators=200, random_state=42,max_depth=10)
    model.fit(X_train, y_train)

    # 予測
    y_pred = model.predict(X_test)

    # 結果の表示
    # print("Confusion Matrix:")
    # print(confusion_matrix(y_test, y_pred))
    # print("\nClassification Report:")
    # print(classification_report(y_test, y_pred))
    # print("---{}---".format(keyword))
    # print ("accuracy")
    # print (model.score(X_test, y_test))
    
    print(f"---{keyword}(rf)---")
    
    
    print("accuracy:", model.score(X_test, y_test))
    
    
    # 混同行列（行: 実ラベル, 列: 予測ラベル）
    
    if heatmap:
        cm = confusion_matrix(y_test, y_pred)
        cm_df = pd.DataFrame(cm)

        plt.figure(figsize=(4,3))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix for {keyword}')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.show()
    return model


# XGBoost用関数

In [ ]:
def load_and_xgboost_by_keyword(keyword, way="all", heatmap=True):

    # --- データ読み込み ---
    df = pd.read_csv(f"../../../../実験/label_beacon/{keyword}_beacon_label_0~6.csv")
    df_label = df[['beacon_label']].copy()
    
    df = preprocess_beacon_rssi(df)
    
    # --- 特徴量選択 ---
    if way == "all":
        X = df
    elif way == "rssi":
        X = df[["810257F7","81025919", "81025B89","810B3B76"]]

    y = df_label['beacon_label']

    # --- train/test ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    # --- XGBoost model ---
    model = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.7,
        objective='multi:softmax',
        eval_metric='mlogloss',
        random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(f"---{keyword} (XGBoost)---")
    print("accuracy:", model.score(X_test, y_test))

    if heatmap:
        cm = confusion_matrix(y_test, y_pred)
        cm_df = pd.DataFrame(cm)
        plt.figure(figsize=(4,3))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
        plt.title(f'XGBoost Confusion Matrix for {keyword}')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.show()

    return model


# k-NN用関数

In [ ]:
def load_and_knn_by_keyword(keyword, way="all", heatmap=True):
    # --- データ読み込み ---
    df = pd.read_csv(f"../../../../実験/label_beacon/{keyword}_beacon_label_0~6.csv")
    df_label = df[['beacon_label']].copy()
    df = preprocess_beacon_rssi(df)

    # --- 特徴量選択 ---
    if way == "all":
        X = df
    elif way == "rssi":
        X = df[["810257F7","81025919", "81025B89","810B3B76"]]

    y = df_label['beacon_label']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    # --- 必須：標準化 ---
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # --- kNN ---
    model = KNeighborsClassifier(
        n_neighbors=5,
        weights='distance',
        metric='manhattan'
    )
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    print(f"---{keyword} (kNN)---")
    print("accuracy:", model.score(X_test_scaled, y_test))

    if heatmap:
        cm = confusion_matrix(y_test, y_pred)
        cm_df = pd.DataFrame(cm)
        plt.figure(figsize=(4,3))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
        plt.title(f'kNN Confusion Matrix for {keyword}')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.show()

    return model, scaler


# アンサンブル学習

In [ ]:
def load_and_ensemble_by_keyword(keyword, way="all", heatmap=True):
    # --- データ読み込み ---
    df = pd.read_csv(f"../../../../実験/label_beacon/{keyword}_beacon_label_0~6.csv")
    df_label = df[['beacon_label']].copy()
    df = preprocess_beacon_rssi(df)
    if way == "all":
        X = df
    elif way == "rssi":
        X = df[["810257F7","81025919", "81025B89","810B3B76"]]

    y = df_label['beacon_label']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    # スケーリング（kNN用）
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # --- モデル ---
    xgb = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.7,
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=42
    )

    knn = KNeighborsClassifier(
        n_neighbors=5,
        weights='distance',
        metric='manhattan'
    )

    # --- Ensemble ---
    model = VotingClassifier(
        estimators=[('xgb', xgb), ('knn', knn)],
        voting='soft',
        weights=[2, 1]   # XGB 比重高め
    )

    # fit は XGB に生データ、kNN に標準化データが必要 → 別処理
    # VotingClassifier は特徴量を共通化する必要があるため
    # XGB も標準化データで訓練する（実害なし）
    model.fit(X_train_s, y_train)

    y_pred = model.predict(X_test_s)

    print(f"---{keyword} (Ensemble: XGB + kNN)---")
    print("accuracy:", model.score(X_test_s, y_test))

    if heatmap:
        cm = confusion_matrix(y_test, y_pred)
        cm_df = pd.DataFrame(cm)
        plt.figure(figsize=(4,3))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Ensemble Confusion Matrix for {keyword}')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.show()

    return model, scaler


In [ ]:
keywords = ['tanaka1','tanaka2','tuji1','tuji2']
keywords = ['kannno1', 'sato1','mori1','mori2','tanaka1','tanaka2','tuji1','tuji2']

for kw in keywords:
    load_and_rondam_forest_by_keyword(kw,heatmap=False)
    load_and_xgboost_by_keyword(kw,heatmap=False)
    load_and_knn_by_keyword(kw,heatmap=False)
    load_and_ensemble_by_keyword(kw,heatmap=False)

In [ ]:
for kw in keywords:
    rf = load_and_rondam_forest_by_keyword(kw, way="rssi", heatmap=False)
    load_and_rondam_forest_by_keyword(kw,way="rssi",heatmap=False)
    load_and_xgboost_by_keyword(kw,way="rssi",heatmap=False)
    load_and_knn_by_keyword(kw,way="rssi",heatmap=False)
    load_and_ensemble_by_keyword(kw,way="rssi",heatmap=False)

# 複数人学習

In [ ]:
def train_cross_multiA_oneB(
        keywords_A,      # ["kanno1","sato2","nishi1"] など
        keyword_B,       # "morita1" など（1人）
        train_ratio=0.3, # B の何％を学習に使うか（ランダム）
        way="all",
        heatmap=False,
        model_type="rf",
        seed=42
    ):
    """
    A複数名を全データで学習し、B1名を train_ratio のみ学習へ混ぜ、
    残りをテストに使う関数。
    """

    # --------- 特徴量選択関数 ---------
    def select_features(df, way):
        if way == "all":
            cols_to_drop = [col for col in df.columns if 'time' in col]
            return df.drop(columns=cols_to_drop, errors='ignore')
        elif way == "rssi":
            return df[["810257F7","81025919","81025B89","810B3B76"]]
        else:
            raise ValueError("way must be 'all' or 'rssi'")

    # --------- A側（複数人）データ読み込み ---------
    XA_list = []
    yA_list = []

    for kw in keywords_A:
        # dfA = pd.read_csv(f"data2/{kw}_beacondata.csv")
        df = pd.read_csv(f"../../../../実験/label_beacon/{kw}_beacon_label_0~6.csv")
        dfA_label =  df[['beacon_label']].copy()
        dfA = df.drop(columns=['beacon_label'], errors='ignore')

        dfA = preprocess_beacon_rssi(dfA)

        XA_list.append(select_features(dfA, way))
        yA_list.append(dfA_label["beacon_label"])

    XA_all = pd.concat(XA_list).reset_index(drop=True)
    yA_all = pd.concat(yA_list).reset_index(drop=True)

    # --------- B側（1名）読み込み ---------
    df = pd.read_csv(f"../../../../実験/label_beacon/{keyword_B}_beacon_label_0~6.csv")
    dfB_label = df[['beacon_label']].copy()
    dfB = df.drop(columns=['beacon_label'], errors='ignore')

    dfB = preprocess_beacon_rssi(dfB)

    XB = select_features(dfB, way)
    yB = dfB_label["beacon_label"]

    # --------- Bデータをランダム分割 ---------
    n = len(XB)
    idx = np.arange(n)

    rng = np.random.default_rng(seed)
    rng.shuffle(idx)

    n_train = int(n * train_ratio)

    train_idx = idx[:n_train]
    test_idx  = idx[n_train:]

    XB_train = XB.iloc[train_idx].reset_index(drop=True)
    yB_train = yB.iloc[train_idx].reset_index(drop=True)

    XB_test  = XB.iloc[test_idx].reset_index(drop=True)
    yB_test  = yB.iloc[test_idx].reset_index(drop=True)

    # --------- A + B_train を学習 ---------
    X_train = pd.concat([XA_all, XB_train], axis=0).reset_index(drop=True)
    y_train = pd.concat([yA_all, yB_train], axis=0).reset_index(drop=True)

    # --------- モデル定義 ---------
    if model_type == "rf":
        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            random_state=42
        )

    elif model_type == "xgb":
        model = XGBClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.7,
            objective="multi:softmax",
            eval_metric="mlogloss",
            random_state=42
        )

    elif model_type == "knn":
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        XB_test = scaler.transform(XB_test)

        from sklearn.neighbors import KNeighborsClassifier
        model = KNeighborsClassifier(
            n_neighbors=5,
            weights="distance",
            metric="manhattan"
        )
    else:
        raise ValueError("model_type must be 'rf', 'xgb', or 'knn'")

    # --------- 学習 ---------
    model.fit(X_train, y_train)

    # --------- 予測 ---------
    y_pred = model.predict(XB_test)

    # --------- 結果 ---------
    print("---------------------",{model_type},"-----------------------")
    print(f"A persons (train): {keywords_A}")
    print(f"B person (train_ratio={train_ratio*100:.1f}%): {keyword_B}")
    print(f"Train size: {len(X_train)}   Test size: {len(XB_test)}")
    print("------------------------------------------------")
    print(f"Accuracy: {model.score(XB_test, yB_test):.4f}")
    
    score = model.score(XB_test, yB_test)

    if heatmap:
        cm = confusion_matrix(yB_test, y_pred)
        cm_df = pd.DataFrame(cm)
        plt.figure(figsize=(4,3))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix (A={len(keywords_A)} people → B={keyword_B})')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.show()

    return model,score


In [ ]:
model_types = ["rf","xgb","knn"]
train_ratios = [0,0.1,0.2,0.3]

# モデルごとにスコアを保持
acc_dict = {mt: [] for mt in model_types}

for mt in model_types:
    for tr in train_ratios:
        model, score = train_cross_multiA_oneB(
            keywords_A=['tanaka1','tanaka2','takagawa1','takagawa2','tuji1'], 
            keyword_B='tuji2',
            train_ratio=tr,
            way="all",
            model_type=mt
        )
        acc_dict[mt].append(score)

# ---- 描画部分 ----
plt.figure(figsize=(8,5))

for mt in model_types:
    plt.plot(train_ratios, acc_dict[mt], marker='o', label=mt)

plt.xlabel("train_ratio")
plt.ylabel("accuracy")
plt.title("Accuracy vs Train Ratio (per model)")
plt.grid(True)
plt.legend()
plt.show()



# 状態遷移を入れる

In [ ]:
# --------------------------
# 特徴量選択
# --------------------------
def select_features(df, way):
    if way == "all":
        cols_to_drop = [col for col in df.columns if 'time' in col]
        return df.drop(columns=cols_to_drop, errors='ignore')
    elif way == "rssi":
        return df[["810257F7", "81025919", "81025B89", "810B3B76"]]
    else:
        raise ValueError("way must be 'all' or 'rssi'")


def predict_with_proba_multiA(
    keywords_A,
    keyword_B,
    way="all",
    model_type="rf",
    seed=42
):
    """
    A 複数名で訓練し、B の全データを softmax 確率として出力する関数。

    戻り値:
      model          学習した分類器
      y_true         B の正解ラベル列 (len = N)
      y_pred         予測ラベル列
      y_proba        予測確率 (N × 7 の行列)
      X_test         Bの特徴量データ
    """

    # --------------------------
    # 1. A の訓練データを準備
    # --------------------------
    XA_list = []
    yA_list = []

    for kw in keywords_A:
        df = pd.read_csv(f"../../../../実験/label_beacon/{kw}_beacon_label_0~6.csv")

        df_label = df[['beacon_label']].copy()
        df_feat  = df.drop(columns=['beacon_label'], errors='ignore')

        df_feat = preprocess_beacon_rssi(df_feat)

        XA_list.append(select_features(df_feat, way))
        yA_list.append(df_label["beacon_label"])

    X_train = pd.concat(XA_list).reset_index(drop=True)
    y_train = pd.concat(yA_list).reset_index(drop=True)

    # --------------------------
    # 2. B のテストデータを準備
    # --------------------------
    df = pd.read_csv(f"../../../../実験/label_beacon/{keyword_B}_beacon_label_0~6.csv")

    dfB_label = df[['beacon_label']].copy()
    dfB_feat  = df.drop(columns=['beacon_label'], errors='ignore')

    dfB_feat = preprocess_beacon_rssi(dfB_feat)

    X_test = select_features(dfB_feat, way).reset_index(drop=True)
    y_test = dfB_label["beacon_label"].reset_index(drop=True)

    # --------------------------
    # 3. モデル定義（train_cross_multiA と同じ）
    # --------------------------
    if model_type == "rf":
        model = RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            random_state=seed
        )

    elif model_type == "xgb":
        model = XGBClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.7,
            objective="multi:softprob",   # softmax 確率を出力
            eval_metric="mlogloss",
            random_state=seed
        )

    elif model_type == "knn":
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        model = KNeighborsClassifier(
            n_neighbors=5,
            weights="distance",
            metric="manhattan"
        )

    else:
        raise ValueError("model_type must be 'rf','xgb','knn'")

    # --------------------------
    # 4. モデル学習
    # --------------------------
    model.fit(X_train, y_train)

    # --------------------------
    # 5. 予測確率を取得
    # --------------------------
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)   # ← HMM の観測確率に渡すコア部分

    return model, y_test, y_pred, y_proba, X_test


In [ ]:
# 7クラス: 0~6
# 隣接クラスは0.8、離れたクラスは0.05（適宜調整）

next = 0.2

T = np.array([
    [1-next, next, 0,   0,   0,   0,   0],
    [next/2, 1-next, next/2, 0,   0,   0,   0],
    [0,   next/2, 1-next, next/2, 0,   0,   0],
    [0,   0,   next/2, 1-next, next/2, 0,   0],
    [0,   0,   0,   next/2, 1-next, next/2, 0],
    [0,   0,   0,   0,   next/2, 1-next, next/2],
    [0,   0,   0,   0,   0,   next, 1-next]
])

def add_floor_and_renorm(T, floor=1e-12):
    T = np.asarray(T, dtype=float)
    T = np.clip(T, 0.0, None) + floor
    T /= T.sum(axis=1, keepdims=True)
    return T

T = add_floor_and_renorm(T, floor=1e-12)


def forward_only_hmm(y_proba, T):
    """
    y_proba : (N, 7) numpy 配列（各フレームの softmax 確率）
    T       : (7,7) 遷移行列
    戻り値：
      belief_seq : (N, 7) 各時刻の状態確率
      pred_class : (N,) 各時刻の推定クラス
    """
    N, n_class = y_proba.shape
    belief_seq = np.zeros_like(y_proba)
    pred_class = np.zeros(N, dtype=int)

    # t=0 の初期状態
    belief_seq[0] = y_proba[0] / y_proba[0].sum()
    pred_class[0] = belief_seq[0].argmax()

    # t>=1 の更新
    for t in range(1, N):
        # 前時刻の belief と遷移行列で予測
        pred = belief_seq[t-1] @ T
        # 観測確率を掛ける
        belief = pred * y_proba[t]
        # 正規化
        belief /= belief.sum()
        belief_seq[t] = belief
        pred_class[t] = belief.argmax()

    return belief_seq, pred_class



def evaluate_with_tolerance(y_true, y_pred, tol=1):
    """
    y_true, y_pred : numpy 配列または pandas Series
    tol             : 隣接クラス許容幅（例: 1なら±1クラスを許容）
    
    戻り値：
        accuracy_tol : 隣接許容精度
        mae          : 平均絶対誤差
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    correct_tol = np.abs(y_true - y_pred) <= tol
    accuracy_tol = correct_tol.mean()

    # 平均絶対誤差
    mae = np.mean(np.abs(y_true - y_pred))

    return accuracy_tol, mae


In [ ]:
all_people = ['tanaka1','tanaka2','takagawa1','takagawa2','tuji1','tuji2','maeda1','maeda2','okawa1','okawa2']

# 1. Aで訓練 → B の softmax 確率を取得
model, y_true, y_pred, y_proba, X_test = predict_with_proba_multiA(
    keywords_A=['maeda2'],
    keyword_B='okawa2',
    way='all',
    # way='rssi',
    model_type='rf'
    # model_type='xgb'
)

# 2. forward-only HMM で平滑化
belief_seq, pred_class = forward_only_hmm(y_proba, T)

# 3. 精度確認
accuracy = (pred_class == y_true.values).mean()
print(f"Forward-only HMM accuracy: {accuracy:.4f}")


plt.figure(figsize=(12,4))
plt.plot(y_pred, label='before Predicted Class')
plt.plot(y_true.values, alpha=0.5, label='True Class')
plt.xlabel('Frame')
plt.ylabel('Class')
plt.legend()
plt.show()


# 4. optional: 確率推移可視化
plt.figure(figsize=(12,4))
plt.plot(pred_class, label='after Predicted Class')
plt.plot(y_true.values, alpha=0.5, label='True Class')
plt.xlabel('Frame')
plt.ylabel('Class')
plt.legend()
plt.show()


In [ ]:
# --- HMM 前 ---
accuracy_before = (y_pred == y_true.values).mean()
accuracy_tol_before, mae_before = evaluate_with_tolerance(y_true.values, y_pred)

print(f"Classifier only accuracy: {accuracy_before:.4f}")
print(f"Classifier only tolerance accuracy (±1): {accuracy_tol_before:.4f}, MAE: {mae_before:.2f}")

# --- forward-only HMM 後 ---
belief_seq, pred_class = forward_only_hmm(y_proba, T)
accuracy_after = (pred_class == y_true.values).mean()
accuracy_tol_after, mae_after = evaluate_with_tolerance(y_true.values, pred_class)

print(f"Forward-only HMM accuracy: {accuracy_after:.4f}")
print(f"Forward-only HMM tolerance accuracy (±1): {accuracy_tol_after:.4f}, MAE: {mae_after:.2f}")


In [ ]:
# all_people = ['tanaka1','tanaka2','tuji1','tuji2']

from itertools import combinations

splits = []
for test_person in all_people:
    train_people = [p for p in all_people if p != test_person]
    splits.append((train_people, test_person))

results = []

model_types = ['rf', 'xgb']

for md in model_types:
    print(f"\n=== Model: {md} ===")

    for train_people, test_person in splits:
        print(f"--- Train: {train_people} | Test: {test_person} ---")

        model, y_true, y_pred, y_proba, X_test = predict_with_proba_multiA(
            keywords_A=train_people,
            keyword_B=test_person,
            # way='rssi',
            way='all',
            model_type=md
        )

        # --- HMM 前 ---
        acc_b = (y_pred == y_true.values).mean()
        acc_tol_b, mae_b = evaluate_with_tolerance(y_true.values, y_pred)

        # # --- HMM 後 ---
        belief_seq, pred_class = forward_only_hmm(y_proba, T)
        acc_a = (pred_class == y_true.values).mean()
        acc_tol_a, mae_a = evaluate_with_tolerance(y_true.values, pred_class)

        results.append({
            'model': md,
            'test_person': test_person,
            'acc_before': acc_b,
            'acc_tol_before': acc_tol_b,
            'mae_before': mae_b,
            'acc_after': acc_a,
            'acc_tol_after': acc_tol_a,
            'mae_after': mae_a
        })


In [ ]:
df = pd.DataFrame(results)

print("\n=== Person-wise results ===")
print(df.groupby(['model','test_person']).mean())

print("\n=== Overall mean performance ===")
print(
    df
    .groupby('model')
    .mean(numeric_only=True)
)
df['acc_gain'] = df['acc_after'] - df['acc_before']
df['mae_gain'] = df['mae_before'] - df['mae_after']  # 小さい方が良い

print("\n=== Gain by person ===")
print(
    df.groupby(['model','test_person'])[['acc_gain','mae_gain']].mean()
)


In [ ]:
def load_person_data(keyword, way):
    df = pd.read_csv(f"../../../../実験/label_beacon/{keyword}_beacon_label_0~6.csv")

    y = df["beacon_label"].reset_index(drop=True)
    X = df.drop(columns=["beacon_label"], errors="ignore")

    X = preprocess_beacon_rssi(X)
    X = select_features(X, way).reset_index(drop=True)

    return X, y


In [ ]:
def predict_with_proba_multi(
    keywords_A,
    keywords_B,
    way="all",
    model_type="rf",
    seed=42
):
    """
    複数人Aで学習し、複数人Bを人別に評価する
    """

    # --------------------------
    # 1. 訓練データ
    # --------------------------
    X_train_list, y_train_list = [], []

    for kw in keywords_A:
        X_kw, y_kw = load_person_data(kw, way)
        X_train_list.append(X_kw)
        y_train_list.append(y_kw)

    X_train = pd.concat(X_train_list).reset_index(drop=True)
    y_train = pd.concat(y_train_list).reset_index(drop=True)

    # --------------------------
    # 2. モデル
    # --------------------------
    if model_type == "rf":
        model = RandomForestClassifier(
            n_estimators=10,
            max_depth=10,
            random_state=seed,
            class_weight='balanced'
        )

    elif model_type == "xgb":
        model = XGBClassifier(
            n_estimators=10,
            learning_rate=0.05,
            max_depth=10,
            subsample=0.8,
            colsample_bytree=0.7,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=seed
        )

    elif model_type == "knn":
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)

        model = KNeighborsClassifier(
            n_neighbors=5,
            weights="distance",
            metric="manhattan"
        )

    else:
        raise ValueError("invalid model_type")

    # --------------------------
    # 3. 学習
    # --------------------------
    model.fit(X_train, y_train)

    # --------------------------
    # 4. 人別テスト
    # --------------------------
    results = {}

    for kw in keywords_B:
        X_test, y_test = load_person_data(kw, way)

        if model_type == "knn":
            X_test = scaler.transform(X_test)

        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        results[kw] = {
            "y_true": y_test,
            "y_pred": y_pred,
            "y_proba": y_proba
        }

    return model, results


In [ ]:
from itertools import combinations

records = []

model_types = ["rf", "xgb"]

for md in model_types:
    print(f"\n===== MODEL: {md} =====")

    for k in range(1, len(all_people)):  # 学習人数
        print(f"\n--- Train size: {k} ---")

        for train_people in combinations(all_people, k):
            test_people = [p for p in all_people if p not in train_people]

            model, results = predict_with_proba_multi(
                keywords_A=list(train_people),
                keywords_B=test_people,
                # way="rssi",
                way="all",
                model_type=md
            )

            for test_person in test_people:
                res = results[test_person]
                y_true  = res["y_true"].values
                y_pred  = res["y_pred"]
                y_proba = res["y_proba"]

                # --- HMM 前 ---
                acc_tol_b, mae_b = evaluate_with_tolerance(y_true, y_pred)

                # --- HMM 後 ---
                belief, pred = forward_only_hmm(y_proba, T)
                acc_tol_a, mae_a = evaluate_with_tolerance(y_true, pred)

                records.append({
                    "model": md,
                    "train_size": k,
                    "train_people": train_people,
                    "test_person": test_person,
                    "acc_tol_before": acc_tol_b,
                    "mae_before": mae_b,
                    "acc_tol_after": acc_tol_a,
                    "mae_after": mae_a
                })

df_results = pd.DataFrame(records)

summary = (
    df_results
    .groupby(["model", "train_size"])
    .agg(
        acc_tol_before_mean=("acc_tol_before", "mean"),
        acc_tol_after_mean=("acc_tol_after", "mean"),
        mae_before_mean=("mae_before", "mean"),
        mae_after_mean=("mae_after", "mean"),
    )
    .reset_index()
)

summary["acc_tol_gain"] = (
    summary["acc_tol_after_mean"] - summary["acc_tol_before_mean"]
)
summary["mae_gain"] = (
    summary["mae_before_mean"] - summary["mae_after_mean"]
)

summary



In [ ]:
# samary をexcelに保存
# summary.to_excel("hmm_multiA_multiB_results.xlsx", index=False)

In [ ]:

for md in model_types:
    df_m = summary[summary["model"] == md]

    plt.figure()
    plt.plot(df_m["train_size"], df_m["acc_tol_before_mean"], marker="o", label="Before")
    plt.plot(df_m["train_size"], df_m["acc_tol_after_mean"], marker="o", label="After")

    plt.xlabel("Number of training persons")
    plt.ylabel("Accuracy")
    plt.title(f"{md}: Accuracy vs train size")
    plt.legend()
    plt.grid(True)
    plt.show()

for md in model_types:
    df_m = summary[summary["model"] == md]

    plt.figure()
    plt.plot(df_m["train_size"], df_m["mae_before_mean"], marker="o", label="Before")
    plt.plot(df_m["train_size"], df_m["mae_after_mean"], marker="o", label="After")

    plt.xlabel("Number of training persons")
    plt.ylabel("MAE")
    plt.title(f"{md}: MAE vs train size")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
plt.figure()
for p in all_people:
    df_p = df_results[
        (df_results["test_person"] == p) &
        (df_results["model"] == "rf")
    ]

    mean_by_k = df_p.groupby("train_size")["acc_tol_after"].mean()
    plt.plot(mean_by_k.index, mean_by_k.values, marker="o", label=p)

plt.xlabel("Train size")
plt.ylabel("acc_tol (After HMM)")
plt.title("RF: person-wise generalization")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 全員の中から一人を訓練に回すクロスバリデーションして、その結果を保持
cv_results = []
model_types = ["rf", "xgb"]

for md in model_types:
    print(f"\n=== Model: {md} ===")

    for train_person in all_people:
        print(f"--- Train person: {train_person} ---")

        test_people = [p for p in all_people if p != train_person]
        model, results = predict_with_proba_multi(
            keywords_A=[train_person],
            keywords_B=test_people,
            # way="rssi",
            way="all",
            model_type=md
        )
        for test_person in test_people:
            res = results[test_person]
            y_true  = res["y_true"].values
            y_pred  = res["y_pred"]
            y_proba = res["y_proba"]

            # --- HMM 前 ---
            acc_tol_b, mae_b = evaluate_with_tolerance(y_true, y_pred)

            # --- HMM 後 ---
            belief, pred = forward_only_hmm(y_proba, T)
            acc_tol_a, mae_a = evaluate_with_tolerance(y_true, pred)

            cv_results.append({
                "model": md,
                "train_person": train_person,
                "test_person": test_person,
                "acc_tol_before": acc_tol_b,
                "mae_before": mae_b,
                "acc_tol_after": acc_tol_a,
                "mae_after": mae_a
            })

# DataFrame 化
df_cv = pd.DataFrame(cv_results)

# モデル・訓練者別に平均精度を表示
order = all_people  # 例: ["tanaka1","tanaka2",...]
order_map = {p: i for i, p in enumerate(order)}

cv_mean = (df_cv
    .groupby(["model", "train_person"], sort=False)
    .mean(numeric_only=True)
    .reset_index()
)

cv_mean["train_order"] = cv_mean["train_person"].map(order_map)

cv_mean = (cv_mean
    .sort_values(["model", "train_order"])
    .drop(columns=["train_order"])
    .set_index(["model", "train_person"])
)

print("\n=== Cross-validation results (reordered) ===")
print(cv_mean)



In [ ]:
# install openpyxl if missing and import it
# %pip install openpyxl

import openpyxl

#  プリントした結果（groupbyした結果）をexcelに保存
# df_summary = (
#     cv_mean
#     .groupby(["model", "train_person"])
#     .mean(numeric_only=True)
#     .reset_index()
# )

# df_summary.to_excel("results.xlsx", index=False)

cv_mean.to_excel("results.xlsx")

In [ ]:
# 訓練者別にHMM前、後の精度を棒グラフでプロット
for md in model_types:
    df_m = df_cv[df_cv["model"] == md]

    x = np.arange(len(all_people))
    width = 0.35

    acc_before = df_m.groupby("train_person")["acc_tol_before"].mean().values
    acc_after  = df_m.groupby("train_person")["acc_tol_after"].mean().values

    plt.figure(figsize=(10,6))
    plt.bar(x - width/2, acc_before, width, label="Before HMM")
    plt.bar(x + width/2, acc_after, width, label="After HMM")

    plt.xticks(x, all_people)
    plt.xlabel("Train Person")
    plt.ylabel("Tolerance Accuracy (acc_tol)")
    plt.title(f"{md}: Cross-validation acc_tol by train person")
    plt.legend()
    plt.grid(axis="y")
    plt.show()


In [ ]:
# 訓練者ごとにテストの人の個別精度を箱ひげ図でプロット
# 日本語ラベル対応のためフォント設定　MS Gothic
plt.rcParams["font.family"] = "MS Gothic"

person_map = {p: f"person{i+1}" for i, p in enumerate(all_people)}
tick_labels = [person_map[p] for p in all_people]

for md in model_types:
    df_m = df_cv[df_cv["model"] == md]

    plt.figure(figsize=(10, 6))

    data = [
        df_m[df_m["train_person"] == p]["acc_tol_after"].values
        for p in all_people
    ]

    plt.boxplot(data, tick_labels=tick_labels)
    plt.xlabel("person (train person)")
    plt.ylabel("隣接許容精度（状態遷移平滑化後）")
    plt.title(f"{md}: 訓練者別 acc_tol 分布（平滑化後）")
    plt.grid(axis="y")
    plt.show()



In [ ]:
# 表示順：all_people をそのまま使う
order = all_people

for md in model_types:
    df_m = df_cv[df_cv["model"] == md].copy()

    # 同じ(train,test)が複数ある場合に備えて平均を取る（上書き防止）
    heatmap_data = (df_m
        .pivot_table(index="train_person",
                     columns="test_person",
                     values="acc_tol_after",
                     aggfunc="mean")
        .reindex(index=order, columns=order)
    )

    # 対角（train==test）は評価対象外なら NaN にして空欄にする
    for p in order:
        if p in heatmap_data.index and p in heatmap_data.columns:
            heatmap_data.loc[p, p] = np.nan

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        heatmap_data,
        annot=True, fmt=".4f",
        cmap="coolwarm",
        vmin=heatmap_data.min().min(), vmax=heatmap_data.max().max(),
        cbar_kws={"label": "acc_after"},
        # ラベルをperson1, person2...に変更
        xticklabels=tick_labels,
        yticklabels=tick_labels
    )

    plt.title(f"{md}: モデルユーザ × ターゲットユーザ 平滑化後Acc_loc", fontsize=12)
    plt.xlabel("ターゲットユーザ", fontsize=10)
    plt.ylabel("モデルユーザ", fontsize=10)
    plt.show()


In [ ]:
for md in model_types:
    df_m = df_cv[df_cv["model"] == md].copy()

    heat = (df_m.pivot_table(index="train_person",
                             columns="test_person",
                             values="acc_tol_after",
                             aggfunc="mean")
                 .reindex(index=all_people, columns=all_people))

    # train==test は元々df_cvに無いはずだが、念のため NaN に
    for p in all_people:
        if p in heat.index and p in heat.columns:
            heat.loc[p, p] = np.nan

    row_mean_from_heat = heat.mean(axis=1, skipna=True)  # ←これが「横方向平均」

    groupby_mean = (df_m.groupby("train_person")["acc_tol_after"].mean()
                      .reindex(all_people))

    check = pd.DataFrame({
        "row_mean_from_heat": row_mean_from_heat,
        "groupby_mean": groupby_mean,
        "diff": row_mean_from_heat - groupby_mean
    })
    print(f"\n=== {md} ===")
    print(check)
    print("max|diff| =", check["diff"].abs().max())


In [ ]:
subjects = ["tanaka1","tanaka2","tuji1","tuji2","okawa1","okawa2","maeda1","maeda2"]

df_context = pd.DataFrame()

# 8人のうち、一人テスト対象、残り訓練者全員で試し、それぞれの結果を保存する
for test_person in subjects:
    train_people = [p for p in subjects if p != test_person]

    model, results = predict_with_proba_multi(
        keywords_A=train_people,
        keywords_B=[test_person],
        way="all",
        model_type="rf"
    )

    res = results[test_person]
    y_true  = res["y_true"].values
    y_pred  = res["y_pred"]
    y_proba = res["y_proba"]

    belief, pred = forward_only_hmm(y_proba, T)

    df_temp = pd.DataFrame({
        "test_person": test_person,
        "real_labels": y_true,
        "y_pred": pred
    })

    df_context = pd.concat([df_context, df_temp], axis=0).reset_index(drop=True)

In [ ]:
# df_context.to_csv("results_context_final_location.csv", index=False)